# 41 Presubmission robustness: harmonized hazard score and field-ranking sensitivity

This notebook resolves the internal inconsistency between the Portugal and Bangladesh hazard formulas. It reuses the frozen endpoint predictions, converts all five endpoints to Development-OOF-referenced hazard percentiles, and applies identical hazard variants to both field cases.

The canonical score in both field cases uses the maximum inverse-toxicity percentile and the geometric mean of P, B and T. Alternative toxicity operators, aggregation operators and endpoint-restricted scores are retained as sensitivity analyses. These analyses demonstrate ranking behavior; they do not validate that unresolved candidates are hazardous.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = ROOT.parent
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = Path(r'E:\谱图预测PBT_终版')
assert (ROOT / 'outputs' / 'pbt_msms_v1').exists(), 'Project root not found.'

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pbt_msms.presubmission_hazard_sensitivity as analysis_module
analysis_module = importlib.reload(analysis_module)
print('ROOT:', ROOT)
print('Python:', sys.executable)
print('Module version:', analysis_module.VERSION)

## Fixed analysis plan

In [ ]:
PLAN = analysis_module.analysis_plan()
display(pd.Series(PLAN, name='fixed_value').to_frame())
display(analysis_module.formula_table())

## Input validation

In [ ]:
input_check = analysis_module.validate_inputs(ROOT)
display(input_check)
assert input_check['exists'].all()
print('PASS: all required frozen field-case inputs are available.')

## Execute

This notebook does not refit an endpoint model. It recalculates scores, ranks, known-target continuity summaries and top-set migration.

In [ ]:
result = analysis_module.run_analysis(ROOT)
display(pd.Series(result['manifest'], name='value').to_frame())

## Rank stability and top-set overlap

In [ ]:
display(result['rank_stability'])

## Portugal known-target continuity

These are descriptive recovery benchmarks, not hazard-positive ground truth.

In [ ]:
display(result['target_recovery'])

## Hazard–occurrence/exposure orthogonality

In [ ]:
display(result['orthogonality'])

## Candidate migration

In [ ]:
migration = result['migration']
display(migration.groupby(['case', 'migration']).size().rename('n').reset_index())
display(migration.loc[migration['migration'].ne('retained')].head(100))

## Quality control and outputs

In [ ]:
display(result['qc'])
assert result['qc']['passed'].astype(bool).all()
print('Summary:', result['summary'])
print('Figure:', result['figure'])
print('Output directory:', result['output_dir'])